In [1]:
import os
import urllib.request
import tarfile
import pandas as pd
from sklearn.model_selection import train_test_split

# === Step 1: Define target base directory ===
base_dir = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data"

# Create necessary subdirectories
images_dir = os.path.join(base_dir, "images")
masks_dir = os.path.join(base_dir, "masks")
annotations_dir = os.path.join(base_dir, "annotations")
os.makedirs(images_dir, exist_ok=True)
os.makedirs(masks_dir, exist_ok=True)
os.makedirs(annotations_dir, exist_ok=True)

# === Step 2: Define dataset URLs ===
images_url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
annotations_url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"

# === Step 3: Download function ===
def download_file(url, dest_path):
    if not os.path.exists(dest_path):
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, dest_path)
        print(f"Saved to {dest_path}")
    else:
        print(f"File {dest_path} already exists. Skipping download.")

# Download the image and annotation archives
images_tar_path = os.path.join(base_dir, "images.tar.gz")
annotations_tar_path = os.path.join(base_dir, "annotations.tar.gz")
download_file(images_url, images_tar_path)
download_file(annotations_url, annotations_tar_path)

# === Step 4: Extract the archives ===
print("Extracting images...")
with tarfile.open(images_tar_path) as tar:
    tar.extractall(path=base_dir)

print("Extracting annotations...")
with tarfile.open(annotations_tar_path) as tar:
    tar.extractall(path=base_dir)

# === Step 5: Move trimap masks to 'masks/' folder ===
trimaps_src_dir = os.path.join(base_dir, "annotations", "trimaps")
for fname in os.listdir(trimaps_src_dir):
    src = os.path.join(trimaps_src_dir, fname)
    dst = os.path.join(masks_dir, fname)
    if not os.path.exists(dst):
        os.rename(src, dst)

# === Step 6: Parse annotation text files ===
def parse_annotation_file(txt_path):
    lines = open(txt_path).readlines()
    records = []
    for line in lines:
        parts = line.strip().split()
        filename = parts[0]
        label = int(parts[1]) - 1  # Class label (0 to 36)
        records.append({
            "filename": filename,
            "image_path": os.path.join(images_dir, f"{filename}.jpg"),
            "mask_path": os.path.join(masks_dir, f"{filename}.png"),
            "label": label
        })
    return records

# Combine trainval and test files to control the full split manually
trainval_txt = os.path.join(base_dir, "annotations", "trainval.txt")
test_txt = os.path.join(base_dir, "annotations", "test.txt")
all_data = parse_annotation_file(trainval_txt) + parse_annotation_file(test_txt)
df_all = pd.DataFrame(all_data)

# === Step 7: Create 70/10/20 split (train/val/test) ===

# First, split 20% for test
df_trainval, df_test = train_test_split(
    df_all,
    test_size=0.2,
    stratify=df_all["label"],
    random_state=42
)

# Then, split 12.5% of the remaining 80% for validation (12.5% of 80% = 10% total)
df_train, df_val = train_test_split(
    df_trainval,
    test_size=0.125,
    stratify=df_trainval["label"],
    random_state=42
)

# === Step 8: Save splits to CSV ===
df_train.to_csv(os.path.join(base_dir, "train.csv"), index=False)
df_val.to_csv(os.path.join(base_dir, "val.csv"), index=False)
df_test.to_csv(os.path.join(base_dir, "test.csv"), index=False)

# === Done ===
print("Dataset preparation completed successfully.")
print(f"Train: {len(df_train)} samples")
print(f"Validation: {len(df_val)} samples")
print(f"Test: {len(df_test)} samples")


Saved to /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/images.tar.gz
Saved to /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/annotations.tar.gz
Extracting images...
Extracting annotations...
Dataset preparation completed successfully.
Train: 5144 samples
Validation: 735 samples
Test: 1470 samples


In [6]:
#!/usr/bin/env python3
"""
inspect_model.py

Inspect a Keras .h5 model: summary, layer counts, param breakdown,
type statistics, FLOPs, architecture diagram, and CSV export.
"""

import os
import csv
from collections import Counter
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

def print_header(title):
    print('\n' + '='*len(title))
    print(title)
    print('='*len(title) + '\n')

def get_layer_params(layer):
    """Return total, trainable, non-trainable param counts for a layer."""
    total = layer.count_params()
    trainable = sum([tf.size(w).numpy() for w in layer.trainable_weights])
    non_trainable = sum([tf.size(w).numpy() for w in layer.non_trainable_weights])
    return total, trainable, non_trainable

def inspect_model(model_path, baseline_path=None, output_csv='model_layers.csv'):
    # 1. Load model
    print_header("1. Loading Model")
    model = load_model(model_path, compile=False)
    print(f"Model loaded from: {model_path}\n")

    # 2. Summary
    print_header("2. Model Summary")
    model.summary()

    # 3. Top-level and nested layer counts
    print_header("3. Layer Counts")
    top_layers = model.layers
    print(f"Total top-level layers: {len(top_layers)}")
    for layer in top_layers:
        if hasattr(layer, 'layers'):
            print(f" - Block '{layer.name}' contains {len(layer.layers)} sub-layers")

    # 4. Per-layer detail and CSV export
    print_header("4. Per-layer Details & CSV Export")
    fieldnames = ['index','name','class','output_shape','total_params',
                  'trainable_params','non_trainable_params','trainable']
    rows = []
    for idx, layer in enumerate(top_layers):
        total, trainable, non_trainable = get_layer_params(layer)
        rows.append({
            'index': idx,
            'name': layer.name,
            'class': layer.__class__.__name__,
            'output_shape': layer.output_shape,
            'total_params': total,
            'trainable_params': trainable,
            'non_trainable_params': non_trainable,
            'trainable': layer.trainable
        })
        print(f"{idx:3d}: {layer.name:20s} [{layer.__class__.__name__:15s}] "
              f"out={layer.output_shape!s:20s} params={total:9d} "
              f"(trainable={trainable:7d}, non-trainable={non_trainable:7d})")

    with open(output_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nLayer details exported to CSV: {output_csv}")

    # 5. Type aggregates
    print_header("5. Layer Type Aggregates")
    classes = [layer.__class__.__name__ for layer in top_layers]
    type_counts = Counter(classes)
    for cls, cnt in type_counts.items():
        print(f"  {cls:20s}: {cnt}")

    # 6. (Optional) Baseline comparison
    if baseline_path:
        print_header("6. Baseline Comparison")
        baseline = load_model(baseline_path, compile=False)
        diff = len(model.layers) - len(baseline.layers)
        print(f"Baseline layers: {len(baseline.layers)}, Current layers: {len(model.layers)}, Extra: {diff}")

    # 7. Compute FLOPs
    print_header("7. FLOPs Calculation")
    # Create concrete function
    fn = tf.function(lambda x: model(x))
    concrete = fn.get_concrete_function(
        tf.TensorSpec([1] + list(model.input_shape[1:]), model.inputs[0].dtype)
    )
    # Convert variables to constants
    frozen_func = convert_variables_to_constants_v2(concrete)
    graph_def = frozen_func.graph.as_graph_def()

    # Profile FLOPs
    with tf.compat.v1.Graph().as_default() as g:
        tf.import_graph_def(graph_def, name='')
        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        flops = tf.compat.v1.profiler.profile(graph=g,
                                              run_meta=run_meta,
                                              cmd='op',
                                              options=opts)
    print(f"Total FLOPs: {flops.total_float_ops:,}")

    # 8. Architecture diagram
    print_header("8. Saving Architecture Diagram")
    diagram_path = 'model_architecture.png'
    plot_model(model, to_file=diagram_path, show_shapes=True, rankdir='TB')
    print(f"Architecture diagram saved to: {diagram_path}\n")

if __name__ == '__main__':
    # --- USER CONFIGURATION ---
    MODEL_PATH = (
        '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
        'polypclassificationmi/code/data/snapshots/all/'
        'hypVSadn_HDall2023_efficientnet_0_regularized0.0_256x256_'
        '1in_nf64_bnTrue_fcdo0.0_balancedTrue_loss_fl_gamma1.0_sgd_5fold0_best.h5'
    )
    # BASELINE_PATH = '/path/to/your/baseline_model.h5'  # Optional
    inspect_model(MODEL_PATH)



1. Loading Model

Model loaded from: /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/code/data/snapshots/all/hypVSadn_HDall2023_efficientnet_0_regularized0.0_256x256_1in_nf64_bnTrue_fcdo0.0_balancedTrue_loss_fl_gamma1.0_sgd_5fold0_best.h5


2. Model Summary

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 efficientnetb0 (Functional)  (None, 1280)             4049571   
                                                                 
 flatten (Flatten)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 2)                 2562      
                                                                 
Total params: 4,052,133
Trainable params

2025-06-10 12:21:37.883650: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2025-06-10 12:21:37.883851: I tensorflow/core/grappler/clusters/single_machine.cc:358] Starting new session
2025-06-10 12:21:37.884616: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1934] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Instructions for updating:
Use `tf.compat.v1.graph_util.tensor_shape_from_node_def_name`

=========================Options=============================
-max_depth                  10000
-min_bytes                  0
-min_peak_bytes             0
-min_residual_bytes         0
-min_output_bytes           0
-min_micros                 0
-min_accelerator_micros     0
-min_cpu_micros             0
-min_params                 0
-min_float_ops              1
-min_occurrence             0
-step                       -1
-order_by                   float_ops
-account_type_regexes       .*
-start_name_regexes         .*
-trim_name_regexes          
-show_name_regexes          .*
-hide_name_regexes          
-account_displayed_op_only  true
-select                     float_ops
-output                     stdout:

==================Model Analysis Report======================

Doc:
op: The nodes are operation kernel type, such as MatMul, Conv2D. Graph nodes belonging to the same type are aggregated

In [11]:
#!/usr/bin/env python3
"""
train_cat_dog.py

Load the exact EfficientNetB0-based snapshot you provided,
remap multi-class labels into binary cat/dog,
continue training on your CSV-defined dataset,
and save best/final models as .h5.
"""

import pandas as pd
import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

# === 1. Configuration ===
MODEL_SNAPSHOT = (
    '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
    'polypclassificationmi/code/data/snapshots/all/'
    'hypVSadn_HDall2023_efficientnet_0_regularized0.0_256x256_'
    '1in_nf64_bnTrue_fcdo0.0_balancedTrue_loss_fl_gamma1.0_sgd_5fold0_best.h5'
)

CSV_TRAIN = '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/train.csv'
CSV_VAL   = '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/val.csv'
CSV_TEST  = '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/test.csv'

IMG_SIZE      = (256, 256)
BATCH_SIZE    = 32
EPOCHS        = 10
BASE_LR       = 1e-3
CAT_DOG_SPLIT = 12   # raw labels 0–11 = cat, 12+ = dog

# === 2. Data pipeline from CSV ===
def load_and_preprocess(path, raw_label):
    # 2.1 Read & decode JPEG
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    # 2.2 Resize & normalize to [0,1]
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    # 2.3 Map raw_label into binary: 0=cat if <CAT_DOG_SPLIT, else 1=dog
    binary = tf.where(raw_label < CAT_DOG_SPLIT, 0, 1)
    binary = tf.cast(binary, tf.int32)
    # 2.4 One-hot encode into [1,0] or [0,1]
    label = tf.one_hot(binary, depth=2)
    return img, label

def make_dataset(csv_file, shuffle=False):
    df     = pd.read_csv(csv_file)
    paths  = df['image_path'].values
    labels = df['label'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(labels), seed=42)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(CSV_TRAIN, shuffle=True)
val_ds   = make_dataset(CSV_VAL)
test_ds  = make_dataset(CSV_TEST)

print(f"Train batches: {len(train_ds)}, Val batches: {len(val_ds)}, Test batches: {len(test_ds)}")

# === 3. Load & compile the model ===
model = load_model(MODEL_SNAPSHOT, compile=False)

optimizer = SGD(learning_rate=BASE_LR, momentum=0.9)
loss_fn   = tfa.losses.SigmoidFocalCrossEntropy(alpha=0.25, gamma=1.0)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
model.summary()

# === 4. Callbacks ===
callbacks = [
    ModelCheckpoint('catdog_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

# === 5. Training ===
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# === 6. Evaluation on test set ===
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")

# === 7. Save final model ===
model.save('catdog_final.h5')
print("Final model saved as 'catdog_final.h5'")


Train batches: 161, Val batches: 23, Test batches: 46
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 efficientnetb0 (Functional)  (None, 1280)             4049571   
                                                                 
 flatten (Flatten)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 2)                 2562      
                                                                 
Total params: 4,052,133
Trainable params: 2,562
Non-trainable params: 4,049,571
_________________________________________________________________
Epoch 1/10
 35/161 [=====>........................] - ETA: 1:02 - loss: 0.3300 - accuracy: 0.6527

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


161/161 [==============================] - ETA: 0s - loss: 0.3203 - accuracy: 0.6742
Epoch 1: val_accuracy improved from -inf to 0.67755, saving model to catdog_best.h5
161/161 [==============================] - 96s 567ms/step - loss: 0.3203 - accuracy: 0.6742 - val_loss: 0.3157 - val_accuracy: 0.6776 - lr: 0.0010
Epoch 2/10
 13/161 [=>............................] - ETA: 1:10 - loss: 0.3092 - accuracy: 0.7067

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


161/161 [==============================] - ETA: 0s - loss: 0.3223 - accuracy: 0.6773
Epoch 2: val_accuracy did not improve from 0.67755
161/161 [==============================] - 88s 546ms/step - loss: 0.3223 - accuracy: 0.6773 - val_loss: 0.3618 - val_accuracy: 0.6776 - lr: 0.0010
Epoch 3/10
148/161 [==========================>...] - ETA: 6s - loss: 0.3219 - accuracy: 0.6763

...

161/161 [==============================] - ETA: 0s - loss: 0.3176 - accuracy: 0.6773
Epoch 5: val_accuracy did not improve from 0.67755
161/161 [==============================] - 89s 553ms/step - loss: 0.3176 - accuracy: 0.6773 - val_loss: 0.3158 - val_accuracy: 0.6776 - lr: 5.0000e-04
Epoch 6/10
104/161 [==================>...........] - ETA: 27s - loss: 0.3202 - accuracy: 0.6719

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


161/161 [==============================] - ETA: 0s - loss: 0.3182 - accuracy: 0.6773
Epoch 6: val_accuracy did not improve from 0.67755
Restoring model weights from the end of the best epoch: 1.
161/161 [==============================] - 89s 552ms/step - loss: 0.3182 - accuracy: 0.6773 - val_loss: 0.3160 - val_accuracy: 0.6776 - lr: 5.0000e-04
Epoch 6: early stopping
18/46 [==========>...................] - ETA: 13s - loss: 0.3146 - accuracy: 0.6806

Corrupt JPEG data: premature end of data segment


46/46 [==============================] - 22s 477ms/step - loss: 0.3158 - accuracy: 0.6776

Test loss: 0.3158, Test accuracy: 0.6776
Final model saved as 'catdog_final.h5'
